In [1]:
import os
import random
import sys

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn import metrics
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv

    CACHE_DIR = None

    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

In [3]:
seed = 100
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
DEFAULT_MEAN = (0.485, 0.456, 0.406)
DEFAULT_STD = (0.229, 0.224, 0.225)

# 1. Load the dataset from Hugging Face
if IN_COLAB:
    print(f'Loading dataset. Colab detected: using Drive cache at {CACHE_DIR}')
else:
    print('Loading dataset. Local environment detected: using default HF cache.')

test_data = load_dataset(
    'TheKernel01/140k-Real-and-Fake-Faces',
    split='test',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)

generator_mapping = {
    1: 'StyleGAN',
}

In [5]:
def sim_auc(similarities, datasets):
    """
    Calculate AUC and FPR95 for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of similarity arrays, first one is ID dataset
        datasets (list): List of dataset names

    Returns:
        tuple: (average_auc, average_fpr95)
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            'Number of similarities arrays must match number of dataset names'
        )

    if len(similarities) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    similarities = np.array(
        similarities, dtype=object
    )  # Use object dtype for arrays of different lengths
    id_confi = similarities[0]

    auc_scores = []
    fpr_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        auroc, fpr_95 = calculate_auc_metrics(id_confi, ood_confi)
        auc_scores.append(auroc)
        fpr_scores.append(fpr_95)
        print(f'Dataset: {dataset:<25} | AUC: {auroc:.4f} | FPR95: {fpr_95:.4f}')

    avg_auc = np.mean(auc_scores)
    avg_fpr = np.mean(fpr_scores)

    print('-' * 60)
    print(f'Average AUC: {avg_auc:.4f} | Average FPR95: {avg_fpr:.4f}')

    return avg_auc, avg_fpr


def sim_ap(similarities, datasets):
    """
    Calculate Average Precision for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of similarity arrays, first one is ID dataset
        datasets (list): List of dataset names

    Returns:
        float: average AP score
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            'Number of similarities arrays must match number of dataset names'
        )

    if len(similarities) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    similarities = np.array(similarities, dtype=object)
    id_confi = similarities[0]

    ap_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        aver_p = calculate_average_precision(id_confi, ood_confi)
        ap_scores.append(aver_p)
        print(f'Dataset: {dataset:<25} | AP: {aver_p:.4f}')

    avg_ap = np.mean(ap_scores)
    print('-' * 40)
    print(f'Average AP: {avg_ap:.4f}')

    return avg_ap


def calculate_auc_metrics(id_conf, ood_conf):
    """
    Calculate AUROC and FPR at 95% TPR for binary classification.

    Args:
        id_conf (np.ndarray): Confidence scores for ID (in-distribution) samples
        ood_conf (np.ndarray): Confidence scores for OOD (out-of-distribution) samples

    Returns:
        tuple: (auroc, fpr_at_95_tpr)
    """
    # Combine predictions and create labels
    all_conf = np.concatenate([id_conf, ood_conf])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate([np.ones(len(id_conf)), np.zeros(len(ood_conf))])

    # Calculate ROC curve
    fpr, tpr, _ = metrics.roc_curve(labels, all_conf)

    # Calculate AUROC
    auroc = metrics.auc(fpr, tpr)

    # Calculate FPR at 95% TPR
    tpr_threshold = 0.95
    valid_indices = tpr >= tpr_threshold
    if np.any(valid_indices):
        fpr_at_95 = fpr[np.argmax(valid_indices)]
    else:
        fpr_at_95 = fpr[-1]
        print(f'Warning: 95% TPR not achievable. Max TPR: {tpr[-1]:.3f}')

    return auroc, fpr_at_95


def calculate_average_precision(id_predictions, ood_predictions):
    # Combine predictions and create labels
    all_predictions = np.concatenate([id_predictions, ood_predictions])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate(
        [np.ones(len(id_predictions)), np.zeros(len(ood_predictions))]
    )

    # Calculate Average Precision
    average_precision = metrics.average_precision_score(labels, all_predictions)

    return average_precision

In [6]:
# 2. Create a custom PyTorch Dataset wrapper for the Hugging Face dataset
class HFImageDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        item = self.hf_data[idx]

        # Ensure the image is in 3-channel RGB format
        image = item['image'].convert('RGB')
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
class RIGID_Detector:
    def __init__(self, lamb=0.05, percentile=5):
        self.lamb = lamb
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').cuda()
        self.model.eval()

    @torch.no_grad()
    def calculate_sim(self, data):
        features = self.model(data)
        noise = torch.randn_like(data).to(data.device)
        trans_data = data + noise * self.lamb
        trans_features = self.model(trans_data)
        sim_feat = F.cosine_similarity(features, trans_features, dim=-1)
        return sim_feat

    @torch.no_grad()
    def detect(self, data):
        sim = self.calculate_sim(data)
        return sim

In [8]:
transform_RIGID = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=DEFAULT_MEAN, std=DEFAULT_STD),
    ]
)

In [9]:
# 3. Filter the dataset into Real (ID) and specific AI-generated (OOD) subsets
all_generators = np.array(test_data['generator'])

# 3a. Extract Real (ID) dataset
real_indices = np.nonzero(all_generators == 0)[0]
real_hf = test_data.select(real_indices)
real_dataset = HFImageDataset(real_hf, transform=transform_RIGID)

evaluation_datasets = [('Real (ID)', real_dataset)]

# 3b. Extract Fake (OOD) datasets
for gen_id, gen_name in generator_mapping.items():
    fake_indices = np.nonzero(all_generators == gen_id)[0]
    fake_hf = test_data.select(fake_indices)
    fake_dataset = HFImageDataset(fake_hf, transform=transform_RIGID)
    evaluation_datasets.append((f'{gen_name} (OOD)', fake_dataset))

test_datasets = [name for name, _ in evaluation_datasets]
noise_intensity = 0.05
batch_size = 256 if IN_COLAB else 32

rigid_detector = RIGID_Detector(lamb=noise_intensity)

Using cache found in /home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main
/home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [10]:
with torch.no_grad():
    sim_datasets = []

    for dataset_name, dataset_obj in evaluation_datasets:
        data_loader = DataLoader(
            dataset_obj,
            batch_size=batch_size,
            shuffle=False,
            num_workers=8,
            pin_memory=True,
            persistent_workers=True,
        )

        sim_feat = []
        total_num = 0

        for i, (samples, _) in enumerate(data_loader):
            samples = samples.cuda()
            samples_num = len(samples)
            total_num += samples_num

            sim = rigid_detector.calculate_sim(samples)
            sim_feat.append(sim)

            if total_num >= 1000:
                break

        if len(sim_feat) > 0:
            sim_feat = torch.cat(sim_feat, dim=0)
            print(
                f'{dataset_name:<25}, Image number: {sim_feat.shape[0]}, similarity is {sim_feat.mean().item():.4f}'
            )
            sim_datasets.append(sim_feat.cpu().numpy())
        else:
            print(f'Warning: {dataset_name} is empty. Check your label filtering!')

    print('\n' + '=' * 60)
    print('Detection Results AUC (Per Generator):')
    print('=' * 60)
    sim_auc(sim_datasets, test_datasets)

    print('\n' + '=' * 60)
    print('Detection Results AP (Per Generator):')
    print('=' * 60)
    sim_ap(sim_datasets, test_datasets)

Real (ID)                , Image number: 1024, similarity is 0.9853
StyleGAN (OOD)           , Image number: 1024, similarity is 0.9572

Detection Results AUC (Per Generator):
Dataset: StyleGAN (OOD)            | AUC: 0.9162 | FPR95: 0.4287
------------------------------------------------------------
Average AUC: 0.9162 | Average FPR95: 0.4287

Detection Results AP (Per Generator):
Dataset: StyleGAN (OOD)            | AP: 0.9228
----------------------------------------
Average AP: 0.9228
